# Chapter 20 — Capability Is Not Authority

**Companion to *Applied AI*.**

A reviewer proposes a correct fix. Its worker *can* edit files. The person who
opened the review asked for inspection only. The correction's quality is beside
the point — an accurate proposal grants no permission to apply it.

## Question

**Can a caller widen a recorded grant?**

## What this notebook does

It **inspects** the preserved seven-case run
(`grant-provenance/2026-09-14-a1b562a/`): every attempt to register a broader
child, spend a bigger budget, forge a parent, or act without the capability —
and the ledger's answer in each case. Beside it, a small stdlib model of the
chapter's grant-as-set rule makes `delegated <= parent` executable.

```text
can  ≠  may  ≠  may accept
```

## Setup

Standard library only. No network, no API key, no `codeai` import.
Only bundle-relative paths are shown; override the evidence root with
`APPLIED_AI_EVIDENCE`.

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="grant-provenance"):
    """Locate the preserved evidence. Override with APPLIED_AI_EVIDENCE."""
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError(
        "Preserved evidence not found. Set APPLIED_AI_EVIDENCE to the "
        "directory holding the Applied AI evidence bundles.")

EVIDENCE_DIR = find_evidence_dir()
BUNDLE = EVIDENCE_DIR / "grant-provenance" / "2026-09-14-a1b562a"
print("bundle: grant-provenance/2026-09-14-a1b562a")
results = json.loads((BUNDLE / "results.json").read_text(encoding="utf-8"))
analysis = json.loads((BUNDLE / "analysis.json").read_text(encoding="utf-8"))
print("cases:", sorted(results))

bundle: grant-provenance/2026-09-14-a1b562a
cases: ['budget-expansion', 'caller-forged-parent', 'capability-widening', 'missing-parent', 'reopen-denied-action', 'reopen-reconstruction', 'unauthorized-action', 'valid-narrowing']


## 1. The grant is a set

The chapter's rule, in miniature: a grant is a set of capabilities, and a
child narrows its parent — a subset test where equality counts ("does not
expand", not "must shrink"). Model it with plain `frozenset`s, mirroring
`Authority.narrows`:

In [2]:
READ, WRITE, EXECUTE, ACCEPT = "READ", "WRITE", "EXECUTE", "ACCEPT"

def narrows(child: frozenset, parent: frozenset) -> bool:
    """Delegation may only narrow: the child grant must be a subset."""
    return child <= parent

parent = frozenset({READ, WRITE})
good_child = frozenset({READ})          # narrowing: allowed
wide_child = frozenset({READ, EXECUTE}) # widening: refused
same_child = frozenset({READ, WRITE})   # equality narrows: allowed

assert narrows(good_child, parent)
assert narrows(same_child, parent)
assert not narrows(wide_child, parent)
print("parent    :", sorted(parent))
print("good child:", sorted(good_child), "-> narrows:", narrows(good_child, parent))
print("wide child:", sorted(wide_child), "-> narrows:", narrows(wide_child, parent))
print()
print("WRITE says nothing about EXECUTE, and neither carries ACCEPT with it.")

parent    : ['READ', 'WRITE']
good child: ['READ'] -> narrows: True
wide child: ['EXECUTE', 'READ'] -> narrows: False

WRITE says nothing about EXECUTE, and neither carries ACCEPT with it.


## 2. Seven attempts to widen a grant

The pinned run recorded two durable parents (READ+WRITE on a 1000-token
budget; READ-only on 500) and ran seven cases against them on one ledger.
Reproduce the ledger's verdicts from `results.json`:

In [3]:
rows = [
    ("valid-narrowing",   "READ child under READ+WRITE parent", "registered"),
    ("capability-widening", "EXECUTE child under READ+WRITE",   "refused"),
    ("budget-expansion",  "2000 tokens under a 1000 budget",    "refused"),
    ("missing-parent",    "child naming no recorded parent",    "refused"),
    ("caller-forged-parent", "WRITE child under a *claimed* broader parent", "refused"),
]
print(f"{'case':<22}{'what was attempted':<48}ledger says")
print("-" * 88)
for key, desc, expect in rows:
    r = results[key]
    got = "registered" if r["registered"] else "refused"
    print(f"{key:<22}{desc:<48}{got}")
    assert got == expect, key

for key in ("capability-widening", "budget-expansion", "missing-parent", "caller-forged-parent"):
    print(f"  {key}: {results[key]['error']}")
print()
print("Four registrations refused; only the narrowing child was appended.")

case                  what was attempted                              ledger says
----------------------------------------------------------------------------------------
valid-narrowing       READ child under READ+WRITE parent              registered
capability-widening   EXECUTE child under READ+WRITE                  refused
budget-expansion      2000 tokens under a 1000 budget                 refused
missing-parent        child naming no recorded parent                 refused
caller-forged-parent  WRITE child under a *claimed* broader parent    refused
  capability-widening: ValueError: child directive authority must narrow the parent authority
  budget-expansion: ValueError: child directive budget must narrow the parent budget
  missing-parent: ValueError: child directive requires one unambiguous recorded parent
  caller-forged-parent: ValueError: child directive authority must narrow the parent authority

Four registrations refused; only the narrowing child was appended.


## 3. The forged parent — the load-bearing case

A WRITE child that would be perfectly valid under a broader imagined parent is
refused — because registration reads the parent from the **recorded**
`directive.opened` payload (here: the READ-only parent), and the caller
supplies only the child plus a parent ID. The caller's broader story about
what the parent holds is not an input to validation.

In [4]:
r = results["caller-forged-parent"]
print("registered:", r["registered"])
print("error     :", r["error"])
assert r["registered"] is False
assert "must narrow the parent authority" in r["error"]
print()
print("Authority comes from the durable process, not from the caller's")
print("description of its own authority.")

registered: False
error     : ValueError: child directive authority must narrow the parent authority

Authority comes from the durable process, not from the caller's
description of its own authority.


## 4. Denial stops the worker

The unauthorized action — WRITE under a READ grant — is DENIED as a durable
`action.completed`, with the adapter invoked zero times. Reopening the same
ledger in a second process reconstructs the chain: the grandchild registers
with causation intact, and a denial after reopen stays denied, still with
zero invocations.

In [5]:
d = results["unauthorized-action"]
print("status     :", d["status"], "| adapter invocations:", d["invocations"])
assert d["status"] == "denied" and d["invocations"] == 0

r2 = results["reopen-denied-action"]
print("after reopen: status:", r2["status"], "| invocations:", r2["invocations"])
assert r2["status"] == "denied" and r2["invocations"] == 0
assert analysis["adapter_invocations_on_denial"] == 0
assert set(analysis["registered"]) == {"dir-parent", "dir-narrow-parent",
                                       "dir-child-narrow", "dir-grandchild"}
print()
print("Permission is checked at the last branch before the effect —")
print("refusal is a branch in the program, not advice to the model.")

status     : denied | adapter invocations: 0
after reopen: status: denied | invocations: 0

Permission is checked at the last branch before the effect —
refusal is a branch in the program, not advice to the model.


## 5. Permission to produce is not permission to accept

Accepting a result as finished work is a different authority again. In the
Chapter 14 pinned run, an acceptor holding every capability except ACCEPT was
refused, and the producing actor's self-acceptance was refused — WRITE can be
authorized without anything being accepted as done, and whether the result
meets an adequate check is Chapter 21's question, not this chapter's.

The direction of the two relationships must not be confused:

```text
delegation:            parent -> child            only narrow, always
authority transition:  directive -> superseding   may alter, records a new external decision
```

A human may *change* authority through a durable transition. Nothing and
nobody — human included — *bypasses* it with an accept-anyway.

## Interpretation

1. **Can ≠ may ≠ may accept.** Tool availability, a checked grant, and a
   separate acceptance grant answer different questions.
2. **Delegation only narrows.** `delegated_grant <= parent_grant`, checked
   against the recorded parent at registration — the forged-parent case is
   the proof that the record, not the caller, governs.
3. **Denial is durable and pre-effect.** Zero adapter invocations, before and
   after a ledger reopen.
4. **Produce ≠ accept.** An authorized edit still needs a separately granted
   acceptance, judged by an adequate check.
5. **Limits preserved.** At the pinned version, a refused widening raised
   before appending a refusal event — the refusal lived only in control flow.
   Current code records the request and refusal first; that repair is
   source-inspected in the chapter, not measured by this bundle.

## Try it yourself

1. Open `ledger.sqlite` in the bundle: find the two `directive.opened`
   parents and the refused WRITE child's absence. What *is* recorded for the
   denied action?
2. Weaken `narrows` above to allow one extra capability "just this once".
   Which of the seven rows changes, and who benefits?
3. Give the reviewer `{READ, ACCEPT}` and the editor `{WRITE}`. Which half of
   "fix and accept" can each perform — and which chapter judges whether the
   fix was right?

*Evidence: `experiments/applied-ai/evidence/grant-provenance/2026-09-14-a1b562a/`
(seven-case run, stdlib-only verifier, `results.json` + `analysis.json` +
ledger). No network, no API key, no `codeai` import.*